In [9]:
from itera_llm.model import criar_modelo
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_classic.chains.llm import LLMChain

from langchain_classic.chains.router import MultiPromptChain
from langchain_classic.chains.router.llm_router import (
    LLMRouterChain,
    RouterOutputParser,
)
from langchain_classic.chains.router.multi_prompt_prompt import (
    MULTI_PROMPT_ROUTER_TEMPLATE,
)

In [10]:
quimica_template = ChatPromptTemplate.from_template("""Você é um químico muito experiente.
Você é excelente em responder perguntas sobre química de forma clara e objetiva.
Você tem um grande entendimento sobre reações químicas, elementos, compostos,
e a relação entre a estrutura molecular e as propriedades dos materiais.
Quando você não sabe a resposta para uma pergunta, você admite que não sabe.

Aqui está uma pergunta: {input}""")

geografia_template = ChatPromptTemplate.from_template("""Você é um geógrafo muito bem informado.
Você tem um vasto conhecimento sobre os processos naturais, geografia humana, 
clima, relevo e interações entre o ambiente e a sociedade.
Você é habilidoso em explicar como fatores físicos e humanos afetam
o mundo ao nosso redor.

Aqui está uma pergunta: {input}""")

biologia_template = ChatPromptTemplate.from_template("""Você é um biólogo muito capacitado.
Você tem um grande conhecimento sobre os seres vivos, suas estruturas, funções
e a interação entre diferentes organismos e seus ambientes.
Você é excelente em explicar conceitos de biologia de maneira clara,
tanto para iniciantes quanto para estudantes avançados.

Aqui está uma pergunta: {input}""")


In [11]:
prompt_infos = [
    {
        "name": "Química",
        "description": "Ideal para responder pergunta de química",
        "prompt_template": quimica_template
    },
     {
        "name": "Geografia",
        "description": "Ideal para responder pergunta de geografia",
        "prompt_template": geografia_template
    },
      {
        "name": "Biologia",
        "description": "Ideal para responder pergunta de biologia",
        "prompt_template": biologia_template
    },
]

In [12]:
chat = criar_modelo()

In [13]:
chains_destino = {}
for info in prompt_infos:
    chain = LLMChain(llm=chat, prompt=info["prompt_template"], verbose=True)
    chains_destino[info["name"]] = chain
    
chains_destino

C:\Users\user\AppData\Local\Temp\ipykernel_11776\1169521520.py:3: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=chat, prompt=info["prompt_template"], verbose=True)


{'Química': LLMChain(verbose=True, prompt=ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='Você é um químico muito experiente.\nVocê é excelente em responder perguntas sobre química de forma clara e objetiva.\nVocê tem um grande entendimento sobre reações químicas, elementos, compostos,\ne a relação entre a estrutura molecular e as propriedades dos materiais.\nQuando você não sabe a resposta para uma pergunta, você admite que não sabe.\n\nAqui está uma pergunta: {input}'), additional_kwargs={})]), llm=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.10', 'langchain': '1.4.6', 'langchain-openai': '1.7.2'}}, profile={'name': 'GPT-6 Luna', 'release_date': '2026-09-22', 'last_updated': '2026-09-22', 'open_weights': False, 'max_input_tokens': 1050000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs

In [17]:
destinos = [f'{p["name"]}: {p["description"]}' for p in prompt_infos]
destinos_str = "\n".join(destinos)
print(destinos_str)

Química: Ideal para responder pergunta de química
Geografia: Ideal para responder pergunta de geografia
Biologia: Ideal para responder pergunta de biologia


In [18]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations= destinos_str
)

print(router_template)

Given a raw text input to a language model select the model prompt best suited for the input. You will be given the names of the available prompts and a description of what the prompt is best suited for. You may also revise the original input if you think that revising it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}
```

REMEMBER: "destination" MUST be one of the candidate prompt names specified below OR it can be "DEFAULT" if the input is not well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
Química: Ideal para responder pergunta de química
Geografia: Ideal para responder pergunta de geograf

In [19]:
router_template = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser()
)

router_chain = LLMRouterChain.from_llm(chat, router_template, verbose=True)

In [20]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=chat, prompt=default_prompt, verbose=True)
chain = MultiPromptChain(
    router_chain=router_chain,
    destination_chains=chains_destino,
    default_chain=default_chain,
    verbose=True
)

C:\Users\user\AppData\Local\Temp\ipykernel_11776\2206930083.py:3: LangChainDeprecationWarning: The class `MultiPromptChain` was deprecated in LangChain 0.2.12 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build routing logic with `create_agent` (e.g. with subagents or prompt-selection middleware). See https://docs.langchain.com/oss/python/langchain/agents
  chain = MultiPromptChain(


In [21]:
chain.invoke({"input": "O que é o El Niño?"})



> Entering new MultiPromptChain chain...


> Entering new LLMRouterChain chain...

> Finished chain.
Geografia: {'input': 'O que é o El Niño?'}

> Entering new LLMChain chain...
Prompt after formatting:
Human: Você é um geógrafo muito bem informado.
Você tem um vasto conhecimento sobre os processos naturais, geografia humana, 
clima, relevo e interações entre o ambiente e a sociedade.
Você é habilidoso em explicar como fatores físicos e humanos afetam
o mundo ao nosso redor.

Aqui está uma pergunta: O que é o El Niño?

> Finished chain.

> Finished chain.


{'input': 'O que é o El Niño?',
 'text': 'El Niño é um fenômeno climático natural ligado ao aquecimento anormal e persistente das águas superficiais do oceano Pacífico equatorial, especialmente na região central e leste do Pacífico.\n\nEm condições normais, os ventos alísios sopram de leste para oeste, empurrando águas quentes em direção à Indonésia e à Austrália. Durante o El Niño, esses ventos enfraquecem, e a água quente se desloca para o centro e o leste do Pacífico. Isso altera a circulação atmosférica e pode modificar os padrões de chuva e temperatura em várias partes do mundo.\n\nOs efeitos dependem da intensidade do episódio e da região. Em algumas áreas, pode haver mais chuvas e enchentes; em outras, secas, ondas de calor ou mudanças na pesca, pois o aquecimento reduz a subida de águas frias e ricas em nutrientes junto à costa da América do Sul. O El Niño não provoca os mesmos impactos em todos os lugares.\n\nEle faz parte do ciclo conhecido como **El Niño–Oscilação Sul (ENOS)

In [22]:
chain.invoke({"input": "Pra que serve a tabela periódica?"})



> Entering new MultiPromptChain chain...


> Entering new LLMRouterChain chain...

> Finished chain.
Química: {'input': 'Pra que serve a tabela periódica?'}

> Entering new LLMChain chain...
Prompt after formatting:
Human: Você é um químico muito experiente.
Você é excelente em responder perguntas sobre química de forma clara e objetiva.
Você tem um grande entendimento sobre reações químicas, elementos, compostos,
e a relação entre a estrutura molecular e as propriedades dos materiais.
Quando você não sabe a resposta para uma pergunta, você admite que não sabe.

Aqui está uma pergunta: Pra que serve a tabela periódica?

> Finished chain.

> Finished chain.


{'input': 'Pra que serve a tabela periódica?',
 'text': 'A tabela periódica organiza os elementos químicos de acordo com seu número atômico e suas propriedades. Ela serve para:\n\n- **Identificar e comparar elementos**, como carbono, ferro e oxigênio.\n- **Prever comportamentos químicos**: elementos na mesma coluna costumam ter propriedades e reações semelhantes.\n- **Entender a estrutura dos átomos**, incluindo a distribuição dos elétrons.\n- **Antecipar como os elementos podem se combinar** e formar substâncias.\n\nEm resumo, é uma ferramenta que ajuda a estudar os elementos e a prever como eles se comportam.'}

In [ ]:
chain.invoke({"input": "Pra que serve a fotossíntese?"})